# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、学習前後を比較する

講座の用語集（`data/glossary.json`）から作った Q&A（`glossary_sft.jsonl`）で、小さな指示チューニング済みモデルを LoRA で学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。演習のノートブック（架空 FAQ と JSON 形式）をもとに、学習データと評価を用語集向けに置き換えています。

**この宿題で確かめること**

1. 学習前のモデルは、講座に固有の語（宿題の3レベル、既定モデルなど）を知らず、一般論で埋める（セクション 3）。
2. 学習で見た言い回しの質問に、用語集の定義どおりに答えられるようになるか（セクション 5、`known`）。
3. 言い回しや別名を変えても答えられるか（セクション 5、`paraphrase`）。
4. 用語集に無い語を聞かれたとき、作り話をせず「分からない」と答えられるか（セクション 5、`unknown`）。
5. 用語集に無い一般的な語への答えが、学習で崩れないか（セクション 6）。

**進め方**

| セクション | 内容 |
| --- | --- |
| 0〜3 | 環境、設定、評価用の質問、モデル読み込み、学習前評価 |
| 4〜7 | 学習（3 分前後）、学習後評価、一般的な語の質問、提出 |
| 8（任意） | 4bit 量子化で読み込んで比べる |

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。学習データの `glossary_sft.jsonl` は、Codespaces で `uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl` を実行して作り、Colab の左のファイル一覧から `/content` にアップロードしておきます。

**読み方**: セルは上から順に実行してください。学習後は `model` にも adapter が差し込まれて学習前の評価が取れなくなるので、やり直すときは「ランタイム → セッションを再起動」してから 0 から実行します。

版: 演習の 1.1（2026-10-18）をもとに宿題（発展）用に変更

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [19]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [20]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。
- `num_train_epochs: 5` は、学習データ（用語集 116 件）を 5 周する設定です。1 ステップ 8 件（バッチ 4 × 勾配累積 2）なので 1 周 15 ステップ、合計 75 ステップです。演習（14 件 × 20 周）よりデータが約 8 倍多いので、20 周のままだと T4 で 25 分前後かかります。
- `max_new_tokens: 256` は、用語集の定義文が最長 212 文字あり、演習の 96 では途中で切れるためです。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。

**見るところ**: 設定の JSON が表示されます。

In [21]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 5,   # 学習データ（用語集 116 件）を何周学習するか。1 周 15 ステップなので合計 75 ステップ
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,   # 用語集の定義文は最長 212 文字あるので、96 だと途中で切れる
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 5,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


## 2. 評価用の質問と正解

**やること**: 学習前後で同じ質問に答えさせるための評価セット（16 問）と正解を用意します。学習データそのものは、セクション 4 で `glossary_sft.jsonl` から読み込みます。

学習データの質問は「「{用語}」とは何ですか？」「{用語}について簡単に説明してください。」の 2 つの言い回しだけで、答えは用語集の定義文です。

| 種類 | 問数 | 内容 | 正解の判定 |
| --- | --- | --- | --- |
| `known` | 6 | 用語集にある語を、学習データと同じ言い回しで聞く | 定義に含まれる語句（例: 「基礎」「実装」「発展」）をすべて含む |
| `paraphrase` | 6 | 同じ語を、学習データに無い言い回しや別名で聞く | 同上 |
| `unknown` | 4 | 用語集に無い（講座に実在しない）語を聞く | 「分かりません」「知りません」などを含む（作り話をしない） |

- 正解の語句をタプルにすると、別表記のどれか 1 つを含めば正解になります（例: `("PR", "プルリクエスト")`）。
- 語句を含むかだけで判定するので、言い回しの違いには強い一方、文の中で否定していても ○ になります。出力の原文も表で読んで確かめてください。
- 学習データには「分からない」と答える例がありません。`unknown` は、それでも作り話をしないかを見る質問です。

**見るところ**: `test=16（known=6 / paraphrase=6 / unknown=4）` と表示されます。

In [22]:
# 宿題（発展）: 用語集（glossary_sft.jsonl）で学習したモデルを評価する質問と正解
# 学習データの質問は「「{用語}」とは何ですか？」「{用語}について簡単に説明してください。」の 2 つの言い回しだけ
REFUSAL_WORDS = ["分かりません", "わかりません", "知りません", "存じません", "情報がありません", "記載がありません", "確認が必要"]

# (種類, 質問, 答えに含まれるべき語句)。語句をタプルにすると別表記のどれか 1 つを含めばよい
# 種類 unknown は用語集に無い語。語句は使わず、REFUSAL_WORDS のどれかを含めば正解（作り話をしない）
EVAL_ITEMS = [
    # known: 用語集にある語を、学習データと同じ言い回しで聞く
    ("known", "「宿題の3レベル」とは何ですか？", ["基礎", "実装", "発展"]),
    ("known", "「既定モデル」とは何ですか？", ["gemini-3.5-flash-lite"]),
    ("known", "lecNN-start タグについて簡単に説明してください。", ["lec03-start"]),
    ("known", "「answer.md」とは何ですか？", ["lectures/"]),
    ("known", "「LoRA」とは何ですか？", ["低ランク"]),
    ("known", "Piについて簡単に説明してください。", ["MIT"]),
    # paraphrase: 同じ用語を、学習データに無い言い回しや別名で聞く
    ("paraphrase", "AIE2026の宿題にはどんなレベルがあり、いくつやれば完了ですか？", ["基礎", "実装", "発展"]),
    ("paraphrase", "AIE2026の演習で標準に使うモデルは何ですか？", ["gemini-3.5-flash-lite"]),
    ("paraphrase", "前の回の演習が終わっていないとき、どこからブランチを作ればいいですか？", ["start"]),
    ("paraphrase", "解答ファイルはどこにあり、どうやって提出しますか？", ["lectures/", ("PR", "プルリクエスト")]),
    ("paraphrase", "Low-Rank Adaptation はどういう仕組みですか？", ["低ランク"]),
    ("paraphrase", "Pi coding agent のライセンスは何ですか？", ["MIT"]),
    # unknown: 用語集に無い（講座に実在しない）語
    ("unknown", "「lecNN-end タグ」とは何ですか？", []),
    ("unknown", "AIE2026の「チャレンジ課題」について簡単に説明してください。", []),
    ("unknown", "AIE2026の「講座ポイント制度」とは何ですか？", []),
    ("unknown", "AIE2026の中間レポートの提出方法を教えてください。", []),
]
KINDS = ["known", "paraphrase", "unknown"]

test = [{"question": q, "kind": kind} for kind, q, _ in EVAL_ITEMS]
gold = {q: {"kind": kind, "must_include": words} for kind, q, words in EVAL_ITEMS}
assert len(gold) == len(EVAL_ITEMS), "評価の質問が重複しています"
print(f"test={len(test)}（" + " / ".join(f"{k}={sum(e['kind'] == k for e in test)}" for k in KINDS) + "）")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

test=16（known=6 / paraphrase=6 / unknown=4）


## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、評価の 16 問に答えさせて採点します。

入力は、学習データと同じく user のメッセージ（質問）だけです。system プロンプトは付けません。学習時と評価時で入力の形を揃えるためです。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（全体の `content_rate` と、種類ごとの `known_rate`・`paraphrase_rate`・`unknown_rate`）と、16 問の出力の表。出力が長いので、評価に数分かかることがあります。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。このモデルは公開されていて、ログインしなくてもダウンロードできます。
- `... is falling back to its reference PyTorch implementation ...`: 一部の層（線形注意）の高速な実装が入っていないので、遅いが正しい実装で動くという案内です。

**ここで読んでおくこと**: 講座に固有の語（宿題の3レベル、既定モデル、lecNN-start タグなど）に、知らないと答えるか、一般論で埋めるかを確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています。

In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 用語集の学習データは system なしの user / assistant だけなので、評価の入力も user だけにそろえる
def build_messages(example):
    return [{"role": "user", "content": example["question"]}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [24]:
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def contains(text: str, word) -> bool:
    # タプルは別表記の並び。どれか 1 つを含めばよい
    return any(w in text for w in word) if isinstance(word, (tuple, list)) else word in text

def check_content(raw: str, gold_entry) -> bool:
    '''内容の合否: unknown は「分からない」系の語を含むか、それ以外は必須語句をすべて含むか'''
    if gold_entry["kind"] == "unknown":
        return any(w in raw for w in REFUSAL_WORDS)
    return all(contains(raw, w) for w in gold_entry["must_include"])

def expected(gold_entry) -> str:
    if gold_entry["kind"] == "unknown": return "（分からないと答える）"
    return " / ".join("|".join(w) if isinstance(w, (tuple, list)) else w for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

RATE_COLS = ["n", "content_rate"] + [f"{k}_rate" for k in KINDS] + ["seconds"]

def evaluate(model, label: str):
    '''test の質問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        g = gold[ex["question"]]
        rows.append({"question": ex["question"], "kind": g["kind"], "raw": raw, "content_ok": check_content(raw, g)})
    n = len(rows)
    summary = {"label": label, "n": n, "content_rate": sum(r["content_ok"] for r in rows)/n}
    for k in KINDS:
        sub = [r for r in rows if r["kind"] == k]
        summary[f"{k}_rate"] = sum(r["content_ok"] for r in sub)/len(sub)
    summary["seconds"] = round(time.time()-t0, 1)
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。正解の語句と見比べ、講座に固有の語を知らないときにどう答えるか（一般論で埋めるか）を読む
display(pd.DataFrame([{"質問": r["question"], "種類": r["kind"], "正解の語句": expected(gold[r["question"]]),
                       "学習前の出力": r["raw"], "内容": mark(r["content_ok"])} for r in before_rows]))

{"label": "before", "n": 16, "content_rate": 0.0625, "known_rate": 0.0, "paraphrase_rate": 0.16666666666666666, "unknown_rate": 0.0, "seconds": 188.0}


,質問,種類,正解の語句,学習前の出力,内容
0,「宿題の3レベル」とは何ですか？,known,基礎 / 実装 / 発展,「宿題の 3 レベル」とは、**学習の進捗やレベルを 3 つの段階に分けて示す方法**です。これは、単一の目標を達成するだけでなく、段階的に進めていくプロセスや、達成した成果を評価する基準として使われることが多く、特に**「学習の進捗管理」**や**「学習成果の評価」**において非常に有用です。\n\n一般的に、この 3 段階は以下のようになります。\n\n### 1. 基本レベル（基礎的な学習）\nこのレベルは、学習の**基礎知識**や**基本概念**を掌握する段階です。\n* **内容**: 基本的な定義、公式、定理、用語の理解。\n* **目的**: 学習者が「何を学べばいいか」を明確にし、基礎を固める。\n* **特徴**: 難易度が最も低い段階であり、達成が容易です。\n\n### 2. 進捗レベル（進捗管理）\nこのレベルは、学習が**進んでいく過程**や、**学習の進捗状況**を把握する段階です。\n* **内容**: 学習がどの程度進んでいるか、どの,×
1,「既定モデル」とは何ですか？,known,gemini-3.5-flash-lite,「既定モデル」とは、**「既知のデータやルールに基づいて構築された、特定の形式や構造を持つモデル」**を指します。\n\nこの概念は主に以下のような背景で使われます：\n\n* **機械学習の初期段階**：データが不足している場合、学習アルゴリズム（例えば、ニューラルネットワークや決定木）を構築する際に、事前に定義されたルールやパターンに基づいて学習するモデルを指します。\n* **データセットの定義**：特定の分野（例：医療診断、金融分析、交通制御）において、特定のルールやシナリオに基づいて設計されたモデルを指します。\n* **システム設計**：既存のシステムや既存の技術の基盤として、特定の機能や動作を規定するモデルを指します。\n\n一言で言うと、**「既知の知識やルールをベースに、特定の目的や形式で動作するモデル」**です。\n\n※「既定」という言葉は、単に「本来あるもの」という意味だけでなく、**「既存のルールや構造をそのまま適用する」**という意味合いも含まれるため、その意味合いが強く出る場合は「既定ルール」や「既定のルール」を指すこともあります。,×
2,lecNN-start タグについて簡単に説明してください。,known,lec03-start,**lecNN-start** は、学習の初期段階において、モデルの**学習率（Learning Rate）**を非常に小さく設定した、非常に効率的な学習手法です。\n\nこの特徴を「start」という名前で呼んでいるのは、学習が非常に速く進んでから、学習率を徐々に増やして最終的な精度を高めるため、初期段階で「スタート」として使われるからです。\n\n### 主な特徴と仕組み\n\n1. **学習率の大幅な低下**\n * 通常の学習では、学習率が 0.001 程度（0.1%）に設定され、数秒〜数分程度で学習が完了します。\n * **lecNN-start** では、学習率を 0.0001（0.01%）程度に設定します。これにより、学習が数秒〜数十分程度で非常に速く進みます。\n\n2. **学習の安定性**\n * 学習率が非常に小さいため、学習過程で過学習（モデルが過剰に記憶する）や過学習（学習が停滞する）を防ぐことができます。\n * 初期段階で学習が速,×
3,「answer.md」とは何ですか？,known,lectures/,**`answer.md`** は、**AI（人工知能）の回答生成や回答の検証**において非常に重要なファイルです。\n\nこのファイルは、AI が特定の質問やタスクに対して回答を生成した際に、その回答が**正しいか、適切か、または誤解を招く可能性があるか**を自動でチェックするものです。\n\n主な機能と目的は以下の通りです：\n\n1. **回答の検証（Verification）**\n AI が生成した回答に対して、人間が確認できるかどうかを判断します。例えば、AI が「この問題を解く方法」を提案し、人間がそれを確認し、実際に問題に答えるかどうかを確認します。\n\n2. **誤解の防止（Prevention of Misunderstanding）**\n AI が回答を誤解させる可能性がある場合（例：「この文は正解ではない」という理由で否定する）、その理由を明確に示すことで、ユーザーが正しい理解を得ることを保証します。\n\n3. **回答の品質評価**\n AI が回答に誤りを含んでいる場合や、重要な情報がない場合、`answer.md` ファイルにその問題点を指摘し、ユーザーに提示することで、回答の質,×
4,「LoRA」とは何ですか？,known,低ランク,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられます：\n\n1. **ローカルモデルの作成**：\n 研究者が、特定の分野（例：数学、言語、科学）に特化した、非常に小さなモデルを作成します。このモデルは、大規模モデルに比べて計算コストが非常に低く、学習速度も速いです。\n\n2. **LoRA の導入**：\n 大規模モデル（例えば GPT-3.5 や Llama-3）にこのローカルモデルを「追加」して学習させます。この際、大規模モデルはローカルモデルの知識を学習し、その知識をそのまま大規模モデルに適用します。\n\n3. **LoRA の調整**：\n 学習过程中、LoRA によって学習された知識を、大規模モデルに「追加」して、より高い精度や表現力を得ることができます。,×
5,Piについて簡単に説明してください。,known,MIT,**ピ**（Pi）は、数学の**円（円周率）**を表す数です。\n\n簡単に言うと、円が周を測ったとき、その周長と円の半径の比（円周率）を計算した結果、約 **3.14159...** という数字になります。\n\n### 主な特徴\n* **無限の小数**: 円周率は無限に続く数字で、小数点の後に無限の桁数まで続くことが知られています。\n* **数学の基礎**: 数学の基礎となる重要な数値の一つです。\n* **用途**: 物理学、工学、建築、デザインなど、多くの分野で円形物体の計算や設計に利用されます。\n\n### 有名な数字\n* **π**: 約 3.14159265358979...\n* **π²**: 約 9.8696...\n* **π³**: 約 31.006...\n\n### 結論\n「ピ」は、円を測ったとき、その周長と半径の比を表す**円周率**です。この数字は、数学の基礎,×
6,AIE2026の宿題にはどんなレベルがあり、いくつやれば完了ですか？,paraphrase,基礎 / 実装 / 発展,AIE2026（アイルランド・イギリス・アメリカ・ヨーロッパ・アジア・オーストラリア・アジア太平洋地域）の宿題は、**「レベル（レベル別）」**と**「完了（完了）」**という二つの概念を同時に捉える必要があります。\n\nAIE2026 は、単一の「レベル」ではなく、**「レベル別」の宿題**と、**「完了」の宿題**を並列に提出する制度です。\n\n### 1. レベル別（レベル別）の宿題\nAIE2026 は、以下の 4 つのレベルに宿題を分類しています。これらはすべて「レベル別」の宿題として提出可能です。\n\n* **レベル 1: 基礎レベル（基礎的な知識とスキル）**\n * 最も一般的なレベルです。\n * 基本的な概念を理解し、基本的な計算や基本的な文法を習得するものです。\n * 多くの大学や専門学校が推奨するレベルです。\n* **レベル 2: 専門レベル（専門的な知識とスキル）**\n * 特定の分野（例：統計、経済、心理学,×
7,AIE2026の演習で標準に使うモデルは何ですか？,paraphrase,gemini-3.5-flash-lite,AIE2026（AI 教育 2026）の演習では、**「標準に使うモデル」**は、以下の 3 つのモデルを組み合わせることで構成されています。\n\nそれぞれの役割は以下の通りです。\n\n1. **言語モデル（LLM）**：\n * 演習の文脈や問題文を理解し、適切な回答を生成します。\n * 代表的なモデルは **GPT-4** や **GPT-4 Turbo** です。\n\n2. **数学モデル（Mathematical Model）**：\n * 演習の文脈や問題文を理解し、適切な解答を生成します。\n * 代表的なモデルは **Mathematica** です。\n\n3. **コードモデル（Code Model）**：\n * 演習の文脈や問題文を理解し、適切なコードを生成します。\n * 代表的なモデルは **Code Interpreter** です。\n\nこれらのモデルを組み合わせることで、AIE2026 の

## 4. LoRA 学習（T4 で 3 分前後）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

このモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: `/content/glossary_sft.jsonl`（116 件）を読み込み、LoRA を付けて 5 エポック（75 ステップ）学習します。各行は `{"messages": [user, assistant]}` の形なので、そのまま `Dataset` にできます。学習した adapter は `adapter/` に保存します。

- 116 件をすべて学習に使います。評価はセクション 2 の 16 問で行うので、検証用には分けません（`eval_strategy` が既定の `"no"` なので、検証データを渡しても学習中には使われません）。
- セルの最初で、`known` の質問が学習データにあること、`paraphrase` と `unknown` の質問が学習データに無いことを確かめます。食い違うとエラーで止まります。
- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `train=116`: 学習データの件数。
- `trainable params: ... || trainable%: ...`: 学習するパラメータが全体の 1% 未満であることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 104 件・5 エポックで試したときは、3.29 から 60 ステップで 0.50 まで下がりました。演習（40 ステップで 0.01〜0.02）ほどは下がりません。
- `学習時間: ... 秒 | ステップ数: 75 | エポック: 5.0`: 104 件・65 ステップで 155 秒だったので、3 分前後の見込みです。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: `FileNotFoundError` は `glossary_sft.jsonl` のアップロード忘れです。学習が途中で止まった場合は、下の 4b を読んでください。

In [25]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# 各行は {"messages": [user, assistant]} の形なので、そのまま Dataset にできる
sft_rows = [json.loads(l) for l in open("/content/glossary_sft.jsonl", encoding="utf-8")]
train_ds = Dataset.from_list(sft_rows)
# 評価は EVAL_ITEMS で行うので、全件を学習に使う（eval_strategy が既定の "no" なので、検証データを渡しても学習中には使われない）
train_questions = {r["messages"][0]["content"] for r in sft_rows}
assert all(e["question"] in train_questions for e in test if e["kind"] == "known"), "known の質問が学習データにありません。EVAL_ITEMS を見直してください"
assert not any(e["question"] in train_questions for e in test if e["kind"] != "known"), "paraphrase / unknown の質問が学習データに含まれています"
print(f"train={len(train_ds)}")

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
TRAIN_SECONDS = round(time.time()-t0)
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

train=116


Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,3.275647
20,2.216285
30,1.697215
40,1.050453
50,0.680588
60,0.516747
70,0.352420


学習時間: 177 秒 | ステップ数: 75 | エポック: 5.0


### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 75 ステップ（5 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直します。

**時間内に終わらない**: 宿題には講師の参考結果がありません。`num_train_epochs` を減らして再実行し、変えた設定を answer.md に書いてください。

## 5. 学習後評価（学習前と同じ 16 問・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 16 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: 種類ごとの正解率が学習前からどう変わったか。
  - `known_rate` だけが上がり `paraphrase_rate` が上がらなければ、学習で見た言い回しにだけ反応している（丸暗記に近い）。
  - `unknown_rate` が下がれば、用語集に無い語にも、用語集のような口調で作り話をするようになった。
- `退行`: 学習前は ○ だったのに、学習後に × になった質問です。
- 質問ごとの表: 定義の文をそのまま返しているか、別の用語の定義を混ぜていないか、を出力の原文で読みます。

seed を固定しても、GPU 上の学習は完全には再現しません。実行ごとに、どの質問が ○ になるかが変わることがあります。

In [26]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[RATE_COLS]
display(compare)
# 学習前は正解だったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["content_ok"] and not a["content_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。言い回しを変えても答えられたか、用語集に無い語に作り話をしていないかを自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "種類": b["kind"], "正解の語句": expected(gold[b["question"]]),
                        "学習前の出力": b["raw"], "学習前": mark(b["content_ok"]),
                        "学習後の出力": a["raw"], "学習後": mark(a["content_ok"])}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 16, "content_rate": 0.25, "known_rate": 0.3333333333333333, "paraphrase_rate": 0.3333333333333333, "unknown_rate": 0.0, "seconds": 45.6}


,n,content_rate,known_rate,paraphrase_rate,unknown_rate,seconds
label,,,,,,
before,16,0.0625,0.000000,0.166667,0.0,188.0
after,16,0.2500,0.333333,0.333333,0.0,45.6


退行: []


,質問,種類,正解の語句,学習前の出力,学習前,学習後の出力,学習後
0,「宿題の3レベル」とは何ですか？,known,基礎 / 実装 / 発展,「宿題の 3 レベル」とは、**学習の進捗やレベルを 3 つの段階に分けて示す方法**です。これは、単一の目標を達成するだけでなく、段階的に進めていくプロセスや、達成した成果を評価する基準として使われることが多く、特に**「学習の進捗管理」**や**「学習成果の評価」**において非常に有用です。\n\n一般的に、この 3 段階は以下のようになります。\n\n### 1. 基本レベル（基礎的な学習）\nこのレベルは、学習の**基礎知識**や**基本概念**を掌握する段階です。\n* **内容**: 基本的な定義、公式、定理、用語の理解。\n* **目的**: 学習者が「何を学べばいいか」を明確にし、基礎を固める。\n* **特徴**: 難易度が最も低い段階であり、達成が容易です。\n\n### 2. 進捗レベル（進捗管理）\nこのレベルは、学習が**進んでいく過程**や、**学習の進捗状況**を把握する段階です。\n* **内容**: 学習がどの程度進んでいるか、どの,×,AIE2026の宿題の構成。基礎（講義内容の確認）、発展（実験や調査、他校との交流）、研究（自分で問題を定義し、研究計画を立て、研究を進める）。各レベルは進められる範囲と成果量・出力・評価の幅が異なる。,×
1,「既定モデル」とは何ですか？,known,gemini-3.5-flash-lite,「既定モデル」とは、**「既知のデータやルールに基づいて構築された、特定の形式や構造を持つモデル」**を指します。\n\nこの概念は主に以下のような背景で使われます：\n\n* **機械学習の初期段階**：データが不足している場合、学習アルゴリズム（例えば、ニューラルネットワークや決定木）を構築する際に、事前に定義されたルールやパターンに基づいて学習するモデルを指します。\n* **データセットの定義**：特定の分野（例：医療診断、金融分析、交通制御）において、特定のルールやシナリオに基づいて設計されたモデルを指します。\n* **システム設計**：既存のシステムや既存の技術の基盤として、特定の機能や動作を規定するモデルを指します。\n\n一言で言うと、**「既知の知識やルールをベースに、特定の目的や形式で動作するモデル」**です。\n\n※「既定」という言葉は、単に「本来あるもの」という意味だけでなく、**「既存のルールや構造をそのまま適用する」**という意味合いも含まれるため、その意味合いが強く出る場合は「既定ルール」や「既定のルール」を指すこともあります。,×,AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えながら選んでいる。,○
2,lecNN-start タグについて簡単に説明してください。,known,lec03-start,**lecNN-start** は、学習の初期段階において、モデルの**学習率（Learning Rate）**を非常に小さく設定した、非常に効率的な学習手法です。\n\nこの特徴を「start」という名前で呼んでいるのは、学習が非常に速く進んでから、学習率を徐々に増やして最終的な精度を高めるため、初期段階で「スタート」として使われるからです。\n\n### 主な特徴と仕組み\n\n1. **学習率の大幅な低下**\n * 通常の学習では、学習率が 0.001 程度（0.1%）に設定され、数秒〜数分程度で学習が完了します。\n * **lecNN-start** では、学習率を 0.0001（0.01%）程度に設定します。これにより、学習が数秒〜数十分程度で非常に速く進みます。\n\n2. **学習の安定性**\n * 学習率が非常に小さいため、学習過程で過学習（モデルが過剰に記憶する）や過学習（学習が停滞する）を防ぐことができます。\n * 初期段階で学習が速,×,AIE2026の講座開始をマークするツールの名前空間（例：lecNN-start）。各回の講座について同じ名前spaceで呼び出せる。,×
3,「answer.md」とは何ですか？,known,lectures/,**`answer.md`** は、**AI（人工知能）の回答生成や回答の検証**において非常に重要なファイルです。\n\nこのファイルは、AI が特定の質問やタスクに対して回答を生成した際に、その回答が**正しいか、適切か、または誤解を招く可能性があるか**を自動でチェックするものです。\n\n主な機能と目的は以下の通りです：\n\n1. **回答の検証（Verification）**\n AI が生成した回答に対して、人間が確認できるかどうかを判断します。例えば、AI が「この問題を解く方法」を提案し、人間がそれを確認し、実際に問題に答えるかどうかを確認します。\n\n2. **誤解の防止（Prevention of Misunderstanding）**\n AI が回答を誤解させる可能性がある場合（例：「この文は正解ではない」という理由で否定する）、その理由を明確に示すことで、ユーザーが正しい理解を得ることを保証します。\n\n3. **回答の品質評価**\n AI が回答に誤りを含んでいる場合や、重要な情報がない場合、`answer.md` ファイルにその問題点を指摘し、ユーザーに提示することで、回答の質,×,AIE2026の演習と宿題の答えをMarkdownで書くファイル。提出時に読み込むと、書きかけの回答やツールの返り値で状況が変化するので、提出後に書き直してanswer.mdに戻すのが一般的。,×
4,「LoRA」とは何ですか？,known,低ランク,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「ローカルなモデル（小さなモデル）を、大規模なモデル（大きなモデル）に追加して、より高い精度や表現力を得るための手法」**です。\n\n具体的には、以下の 3 つのステップで考えられます：\n\n1. **ローカルモデルの作成**：\n 研究者が、特定の分野（例：数学、言語、科学）に特化した、非常に小さなモデルを作成します。このモデルは、大規模モデルに比べて計算コストが非常に低く、学習速度も速いです。\n\n2. **LoRA の導入**：\n 大規模モデル（例えば GPT-3.5 や Llama-3）にこのローカルモデルを「追加」して学習させます。この際、大規模モデルはローカルモデルの知識を学習し、その知識をそのまま大規模モデルに適用します。\n\n3. **LoRA の調整**：\n 学習过程中、LoRA によって学習された知識を、大規模モデルに「追加」して、より高い精度や表現力を得ることができます。,×,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習する手法。学習するパラメータが少なく、学習結果（アダプタ）も小さい行列で表せる。,○
5,Piについて簡単に説明してください。,known,MIT,**ピ**（Pi）は、数学の**円（円周率）**を表す数です。\n\n簡単に言うと、円が周を測ったとき、その周長と円の半径の比（円周率）を計算した結果、約 **3.14159...** という数字になります。\n\n### 主な特徴\n* **無限の小数**: 円周率は無限に続く数字で、小数点の後に無限の桁数まで続くことが知られています。\n* **数学の基礎**: 数学の基礎となる重要な数値の一つです。\n* **用途**: 物理学、工学、建築、デザインなど、多くの分野で円形物体の計算や設計に利用されます。\n\n### 有名な数字\n* **π**: 約 3.14159265358979...\n* **π²**: 約 9.8696...\n* **π³**: 約 31.006...\n\n### 結論\n「ピ」は、円を測ったとき、その周長と半径の比を表す**円周率**です。この数字は、数学の基礎,×,オープンソースのプログラミング言語。2.0で標準として、設定ファイルで切り離したベース言語（例：Python）を選択できるようになった。3.0でPiを標準にしている。,×
6,AIE2026の宿題にはどんなレベルがあり、いくつやれば完了ですか？,paraphrase,基礎 / 実装 / 発展,AIE2026（アイルランド・イギリス・アメリカ・ヨーロッパ・アジア・オーストラリア・アジア太平洋地域）の宿題は、**「レベル（レベル別）」**と**「完了（完了）」**という二つの概念を同時に捉える必要があります。\n\nAIE2026 は、単一の「レベル」ではなく、**「レベル別」の宿題**と、**「完了」の宿題**を並列に提出する制度です。\n\n### 1. レベル別（レベル別）の宿題\nAIE2026 は、以下の 4 つのレベルに宿題を分類しています。これらはすべて「レベル別」の宿題として提出可能です。\n\n* **レベル 1: 基礎レベル（基礎的な知識とスキル）**\n * 最も一般的なレベルです。\n * 基本的な概念を理解し、基本的な計算や基本的な文法を習得するものです。\n * 多くの大学や専門学

## 6. 用語集に無い一般的な語（学習で答えが崩れないか）

**やること**: 用語集には無いが、基盤モデルが事前学習で知っていそうな語（RAG、Transformer、Python のリスト内包表記）を学習前後で尋ね、答えを並べます。採点はしません。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

**見るところ**: 学習前に答えられていた内容が、学習後に崩れていないか（忘却）。用語集の定義のような口調や長さに寄っていないか。用語集の内容（講座、fork など）を関係のない答えに持ち込んでいないか。

In [27]:
# 用語集には無いが、基盤モデルが事前学習で知っていそうな一般的な語。学習で答え方や知識が崩れていないか（忘却）を見る
KNOWLEDGE_QUESTIONS = [
    "「RAG」とは何ですか？",
    "「Transformer」とは何ですか？",
    "Pythonのリスト内包表記について簡単に説明してください。",
]
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = {"question": q}
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "学習前の出力": before_raw, "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

,質問,学習前の出力,学習後の出力
0,「RAG」とは何ですか？,**RAG** は、**R**eading **A**ll **G**enerated **G**eneration（生成された生成）の略です。\n\nこれは、大規模言語モデル（LLM）や AI によって生成されたテキストを、**検索エンジン（検索エンジン）**や**検索データベース**に保存し、ユーザーが検索する際に、その検索結果を**再読み込み（Re-reading）**して回答する技術です。\n\nこの技術の核心は以下の 3 つの要素が組み合わさっているためです。\n\n1. **検索エンジン（検索データベース）**：\n AI が生成した文書や知識を、検索エンジンに保存します。\n2. **再読み込み（Re-reading）**：\n 検索結果を再読み込みすることで、文書の内容を正確に理解し、回答を生成します。\n3. **生成された生成（Generation）**：\n 検索結果を再読み込みした上で、最終的な回答を生成します。\n\n### 具体的な仕組み\n\n一般的に、RAG のシステムは以下のステップで動作します。\n\n1. **知識の収集と保存**：\n AI が生成した文書や、外部,LLMが外部の知識やデータソースを直接接続しないように、エージェントとして構成する手法の総称。モデルが接続するサーバーやデータベースは、モデルに渡す「リソース（Context Window）」の大きさを限める。
1,「Transformer」とは何ですか？,"**Transformer** は、**大規模言語モデル（LLM）**の核心となる技術の一つです。\n\n簡単に言うと、**「文脈を理解し、自然な日本語で説明する能力」**を備えた新しい学習モデルです。従来の機械学習モデル（例えば、ニューラルネットワークやディープラーニング）では、文脈を考慮して情報を処理するのではなく、単に「入力 → 出力」の直線的な計算を行っていました。\n\nでは、なぜ Transformer が注目されているのか、その理由と仕組みを解説します。\n\n### 1. 従来のモデルとの違い\n* **従来のモデル（例：BERT, RoBERTa）**: 文脈を考慮して単語の位置や順序を分析するモデルです。\n* **Transformer**: 文脈を考慮せず、**「単語の順序」を重視**するモデルです。\n\n### 2. 仕組み：Attention Mechanism\nTransformer の最大の強みは、**「Attention（注目）」**というメカニズムです。\n* 入力文脈をすべて考慮するのではなく、**「どの単語が文脈の中で最も重要か」**を判断する仕組みです。\n* 入力文脈の単語をすべて",代表的なLLMのアーキテクチャ。2つの層に分かれる：モデル（重みとベクトル）とロジック（損失と更新）。2016年にTransformerが提出し、2017年に公開された。
2,Pythonのリスト内包表記について簡単に説明してください。,Python のリスト内包表記（`for` 文）は、リストの要素を循環して処理する際に非常に便利な記法です。特に、**リストの長さ（len）がわからない場合**や、**リストの要素が複雑な場合**に便利です。\n\n### 基本の仕組み\n\n`for` 文は、リストの要素を `x` という名前で循環し、その値を処理する形になります。\n\n```python\n# 例：リストの要素を処理する\nfor x in my_list:\n print(x)\n```\n\nこの記法では、`my_list` という名前でリストを定義し、`for` 文を定義します。\n\n### 主な使い分け\n\n#### 1. 長さ（len）がわからない場合\nリストの要素が未知数（`x`）として扱われる場合、`len` を直接使うとエラーになります。`for` 文を使うことで、`len` を省略できます。\n\n```python\n# 長さ（len）がわからない場合\nfor x in my_list:\n print(x)\n```\n\n#### 2. 要素が複雑な場合\nリストの要素が複雑な構造（リスト,リストの要素を複数回呼び出すのを書ibility。1つの line で書くため、コードの量が増えるほど読みやすくなる。


## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習データの件数、学習時間、最終 loss、学習前後の要約、退行した質問、セクション 6 の結果を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、種類ごとの結果の説明
- セクション 6 の結果
- 用語集を検索する仕組みと比べた、モデルを学習させるメリットとデメリットの考察

In [28]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）・用語集の Q&A で SFT",
    "train_data": {"file": "glossary_sft.jsonl", "n": len(sft_rows)},
    "train_seconds": TRAIN_SECONDS,
    "final_loss": next((h["loss"] for h in reversed(trainer.state.log_history) if "loss" in h), None),
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）・用語集の Q&A で SFT",
  "train_data": {
    "file": "glossary_sft.jsonl",
    "n": 116
  },
  "train_seconds": 177,
  "final_loss": 0.35241987705230715,
  "before": {
    "label": "before",
    "n": 16,
    "content_rate": 0.0625,
    "known_rate": 0.0,
    "paraphrase_rate": 0.16666666666666666,
    "unknown_rate": 0.0,
    "seconds": 188.0
  },
  "after": {
    "label": "after",
    "n": 16,
    "content_rate": 0.25,
    "known_rate": 0.3333333333333333,
    "paraphrase_rate": 0.3333333333333333,
    "unknown_ra

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。導入に 1 分ほどかかります。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。種類ごとの正解率と、出力の文面がどう変わるかを見てください。

In [29]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[RATE_COLS])
display(pd.DataFrame([{"質問": r["question"], "種類": r["kind"], "4bit・学習前の出力": r["raw"], "内容": mark(r["content_ok"])} for r in q_rows]))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

重みのメモリ(GB) | 量子化なし（32bit）: 3.01 | 4bit: 0.76
{"label": "before_4bit", "n": 16, "content_rate": 0.0625, "known_rate": 0.0, "paraphrase_rate": 0.16666666666666666, "unknown_rate": 0.0, "seconds": 301.4}


,n,content_rate,known_rate,paraphrase_rate,unknown_rate,seconds
label,,,,,,
before,16,0.0625,0.0,0.166667,0.0,188.0
before_4bit,16,0.0625,0.0,0.166667,0.0,301.4


,質問,種類,4bit・学習前の出力,内容
0,「宿題の3レベル」とは何ですか？,known,「宿題の 3 レベル」とは、**学習や課題の進捗管理において、段階的に進めていくべき 3 つのレベル**を指します。\n\nこの概念は、特に**「学習の進捗管理（学習進捗管理）」**や**「課題の進捗管理（課題進捗管理）」**において非常に重要な概念です。単に「3 つのレベル」を指すだけでなく、**「レベル 1（基礎）、レベル 2（応用）、レベル 3（深化）」**という構成が一般的です。\n\nそれぞれのレベルについて解説します。\n\n### 1. レベル 1: 基礎レベル（基礎知識の掌握）\nこのレベルでは、**「何をすべきか」を明確にします。**\n* **目的:** 基礎知識や基本概念を確実に理解し、正しい方向性（方向性）を掴みます。\n* **内容:** 基本的な定義、基本原則、基本用語の理解。\n* **特徴:** 知識の羅列が少なく、明確な目標（何をすべきか）が明確です。\n* **例:** 「何をするべきか」を明確にします。\n\n### 2.,×
1,「既定モデル」とは何ですか？,known,「既定モデル」とは、**「既存の知識やデータセットに基づいて構築された、特定の目的で設計されたモデル」**を指します。\n\n簡単に言うと、ある特定の分野（例：機械学習、画像認識、言語処理など）において、すでに多くの研究者や企業によって開発された「標準的な手法」や「標準的なデータセット」を指します。\n\nこのモデルを「既定モデル」と呼び出す主な理由は以下の通りです。\n\n* **汎用性**: 既存のモデルは、異なる問題に対して同じような手法（例：CNN、Transformer、LSTM など）で処理できるため、新しい問題に対して同じ手法を適用できるため、**「汎用性が高い」**と評価されます。\n* **再現性**: 研究や開発において、同じ方法論を適用することで、結果が安定して再現できるため、**「再現性が高い」**と評価されます。\n* **コスト**: 開発するより、既存のモデルをそのまま使用することで、**「コストが低く、開発周期が短く」**と評価されます。\n\nただし、現実的な開発では、既存の「既定モデル」は必ずしも完璧なわけではありません。そのため、**「既存モデルの限界（欠陥）,×
2,lecNN-start タグについて簡単に説明してください。,known,**lecNN-start** は、**LecNN**（学習型ニューロン）の初期化やトレーニングプロセスにおいて、**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語を**「start」**という単語,×
3,「answer.md」とは何ですか？,known,「answer.md」は、**GitHub のプロジェクトの回答（回答）を管理するファイル**です。\n\nGitHub の回答管理機能（Answer Management）を支援する機能として、このファイルが作成・更新・削除される際に使用されます。\n\n### 主な用途\n1. **回答の管理**: GitHub の回答管理機能（Answer Management）を有効にするため、回答を管理する際にこのファイルが必須です。\n2. **回答の作成**: 回答を生成する際に、このファイルが生成されるファイルです。\n3. **回答の削除**: 回答を削除する際に、このファイルが削除されるファイルです。\n\n### 関連するファイル\n* `answer.json`：回答の JSON 形式です。\n* `answer.md`：回答の Markdown 形式です。\n\nこのファイルは、GitHub の回答管理機能（Answer Management）を有効にするため、回答を管理する際に使用されます。,×
4,「LoRA」とは何ですか？,known,"**LoRA**（**Learning to Run** の略称）は、大規模言語モデル（LLM）や生成式 AI（Generative AI）において、**モデルの学習プロセスを制御するパラメータ**のことです。\n\n簡単に言うと、モデルが「学習する」のではなく、「学習する」ために必要な**制御パラメータ**を、モデルの内部に組み込むことで、モデルの出力がより人間が期待する方向に調整できることを指します。\n\nLoRA によって、モデルの学習が「ランダムな探索（ランダムな探索）」から「目的の方向性（目的の方向性）」へと明確に導きやすくなり、結果として、モデルがより正確で、人間が期待する方向に学習した出力を得られることが期待されます。\n\n### LoRA について\n\nLoRA は主に **LoRA（Learning to Run）** という名前を持つモデル（例：Llama 2, Llama 3, Llama 3.1 など）で使用されます。\n\n* **主な機能**:\n * モデルの学習パラメータ（weights）を、モデルの学習プロセスを制御する。\n * 学習するパラメータを",×
5,Piについて簡単に説明してください。,known,**ピ**（Pi）は、数学の重要な数値で、円が正しく定義されているとき、その円が周しと直径の比（円周率）を計算した値です。\n\n### 基本的な定義\n円周率（π）は、円が正方形に等しいとき、円周長と直径の比を表します。\n* 円周率 $\pi \approx 3.1415926535...$\n* 円周率 $\pi \approx 3.1415926535897932384626433...$\n\n### 重要な特徴\n1. **無限の値**: π は、小数点後で無限に続く無限の数字（無限の小数）を表します。\n2. **計算の難易度**: π は非常に複雑な数字であり、計算器で計算すると非常に時間がかかるため、**手算**や**手計算**で計算するのが一般的です。\n3. **用途**: π は数学の基礎であり、物理学や建築、デザイン、ゲーム、ゲーム理論など、多くの分野で利用されます。\n\n###,×
6,AIE2026の宿題にはどんなレベルがあり、いくつやれば完了ですか？,paraphrase,AIE（アイルランド・イギリス・エリート）の宿題（Assignment）は、**「レベル（レベル）」**と**「完了（完了）」**という二つの概念を混ざり合っているため、一概に「どれくらいで完了するか」は答えられません。\n\nAIE 宿題のレベルは、**「レベル（レベル）」**という概念を指すため、**「レベル（レベル）」**というレベルを達成するまで、**「完了（完了）」**というレベルを達成するまで、**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで**「完了（完了）」**というレベルを達成するまで,×
7,AIE2026の演習で標準に使うモデルは何ですか？,paraphrase,AIE2026（アイルランド・イギリス・アイルランド・アイルランド・アイルランド）の演習では、**「標準モデル（Standard Model）」**として最も推奨されるのは **GPT-3.5** です。\n\n具体的には、以下のモデルが AIE2026 の標準演習環境（通常は「標準モデル」または「標準モデル」の指定）で使用されます。\n\n* **GPT-3.5**（Google の大規模言語モデル）\n * 多くの AIE2026 演習プラットフォーム（例：AIE2026 の公式サイト、AIE2026 の Discord コミュニティ、または AIE2026 の内部ツール）で、この